# Imports and dependencies

In [ ]:
!pip install spacy transformers -q
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import ast
import spacy
from tqdm import tqdm
import re
import random
import shutil
import json
from transformers import pipeline
import random

In [ ]:
nlp_lg = spacy.load("en_core_web_lg")

# Dataset

In [ ]:
df = pd.read_csv("02_processed.csv")

In [ ]:
df.head()

,id,url,title,text,lemmas,sentences
0,765,https://en.wikipedia.org/wiki/Abortion,Abortion,Abortion is the termination of a pregnancy by ...,"['abortion', 'termination', 'pregnancy', 'remo...",['Abortion is the termination of a pregnancy b...
1,1023,https://en.wikipedia.org/wiki/Anarcho-capitalism,Anarcho-capitalism,Anarcho-capitalism (colloquially: ancap or an-...,"['anarcho', 'capitalism', 'colloquially', 'anc...",['Anarcho-capitalism (colloquially: ancap or a...
2,1921,https://en.wikipedia.org/wiki/Al-Qaeda,Al-Qaeda,Al-Qaeda is a pan-Islamist militant organizati...,"['al', 'qaeda', 'pan', 'islamist', 'militant',...",['Al-Qaeda is a pan-Islamist militant organiza...
3,20979,https://en.wikipedia.org/wiki/Mikhail_Gorbachev,Mikhail Gorbachev,Mikhail Sergeyevich Gorbachev (2 March 1931 – ...,"['mikhail', 'sergeyevich', 'gorbachev', '2', '...","[""Mikhail Sergeyevich Gorbachev (2 March 1931 ..."
4,21023,https://en.wikipedia.org/wiki/Mormonism,Mormonism,Mormonism is the theology and religious tradit...,"['mormonism', 'theology', 'religious', 'tradit...",['Mormonism is the theology and religious trad...


In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(2).to_string())

(80, 6)
['id', 'url', 'title', 'text', 'lemmas', 'sentences']
id            int64
url          object
title        object
text         object
lemmas       object
sentences    object
dtype: object
     id                                               url               title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
df['lemmas'] = df['lemmas'].apply(ast.literal_eval)
df['sentences'] = df['sentences'].apply(ast.literal_eval)

# Named entities extraction

In [ ]:
def extract_entities(text):
    doc = nlp_lg(text)
    entities = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG", "GPE", "NORP", "LOC"}:
            entities.append({
                "text": ent.text,
                "label": ent.label_
            })
    return entities

tqdm.pandas()
df['entities'] = df['text'].progress_apply(extract_entities)

In [ ]:
print(df[['title', 'entities']].head(3).to_string())
print("\nTotal extracted entities per article (top 5):")
print(df['entities'].apply(len).head())

                title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [ ]:
def get_entity_counts(entities):
    """Normalizes text and counts frequencies per type."""
    counts = {}
    for ent in entities:
        label = ent['label']
        # Normalization: lowercase y strip
        text = ent['text'].strip().lower()
        key = (text, label)
        counts[key] = counts.get(key, 0) + 1
    # Ordering by freq
    return sorted(counts.items(), key=lambda x: x[1], reverse=True)

df['entity_counts'] = df['entities'].apply(get_entity_counts)

# Top entities in first 3 articles
for i in range(3):
    print(f"\n--- {df['title'][i]} ---")
    print("Top 15 entities:")
    for (text, label), count in df['entity_counts'][i][:15]:
        print(f"  [{label}] {text}: {count}")


--- Abortion ---
Top 15 entities:
  [GPE] the united states: 18
  [GPE] us: 7
  [GPE] china: 7
  [GPE] u.s.: 6
  [ORG] the world health organization: 5
  [GPE] india: 5
  [ORG] d&e: 5
  [GPE] switzerland: 3
  [GPE] canada: 3
  [ORG] mva: 3
  [GPE] sweden: 3
  [ORG] cdc: 3
  [ORG] the guttmacher institute: 3
  [ORG] the catholic church: 3
  [NORP] catholic: 3

--- Anarcho-capitalism ---
Top 15 entities:
  [PERSON] rothbard: 80
  [NORP] american: 17
  [NORP] anarchist: 15
  [PERSON] murray rothbard: 14
  [PERSON] tucker: 12
  [PERSON] friedman: 11
  [PERSON] david: 8
  [PERSON] spooner: 8
  [PERSON] murray: 7
  [NORP] austrian: 6
  [ORG] mises institute: 6
  [NORP] anarchists: 6
  [PERSON] davis: 6
  [ORG] state: 5
  [ORG] austrian school: 4

--- Al-Qaeda ---
Top 15 entities:
  [ORG] al-qaeda: 212
  [PERSON] bin laden: 57
  [GPE] u.s.: 46
  [ORG] al-qaeda's: 42
  [NORP] islamic: 26
  [NORP] muslim: 24
  [GPE] us: 24
  [LOC] kashmir: 24
  [NORP] american: 23
  [GPE] afghanistan: 21
  [GP

In [ ]:
NORMALIZATION_MAP = {
    "u.s.": "united states",
    "us": "united states",
    "the united states": "united states",
    "u.s": "united states",
    "uk": "united kingdom",
    "the united kingdom": "united kingdom",
    "al-qaeda's": "al-qaeda",
    "al-qaida": "al-qaeda",
    "al-qa'ida": "al-qaeda",
    "el-qaida": "al-qaeda",
    "murray rothbard": "rothbard",
    "murray rothbard's": "rothbard",
    "murray n. rothbard": "rothbard",
    "murray n.": "rothbard",
    "murray": "rothbard",
}

# Pares (texto, label) que son errores de NER conocidos. Se descartan antes de contar.
EXCLUDED_ENTITIES = {
    ("d&e", "ORG"),
    ("mva", "ORG"),
    ("state", "ORG"),
    ("the state", "ORG"),
    ("islam", "ORG"),
}

def fold_plural(text, label):
    """Pliega plural -> singular, solo para NORP. Umbral >3 caracteres para no
    afectar tokens cortos, y excluye terminaciones en 'ss' (p.ej. 'swiss')."""
    if label == "NORP" and len(text) > 3 and text.endswith("s") and not text.endswith("ss"):
        return text[:-1]
    return text

def normalize_entity(text, label=None):
    text = text.strip().lower()
    text = re.sub(r"'s$", "", text)
    text = NORMALIZATION_MAP.get(text, text)
    if label is not None:
        text = fold_plural(text, label)
    return text

def get_entity_counts_normalized(entities):
    counts = {}
    for ent in entities:
        label = ent['label']
        text = normalize_entity(ent['text'], label)
        key = (text, label)
        if key in EXCLUDED_ENTITIES:
            continue
        counts[key] = counts.get(key, 0) + 1
    return sorted(counts.items(), key=lambda x: x[1], reverse=True)

def get_surface_forms(entities):
    forms = {}
    for ent in entities:
        label = ent['label']
        canon = normalize_entity(ent['text'], label)
        key = (canon, label)
        if key in EXCLUDED_ENTITIES:
            continue
        forms.setdefault(key, set()).add(ent['text'].strip())
    return forms

df['entity_counts_norm'] = df['entities'].apply(get_entity_counts_normalized)

# Verification
for i in [0, 2]:
    print(f"\n--- {df['title'][i]} ---")
    for (text, label), count in df['entity_counts_norm'][i][:15]:
        print(f"  [{label}] {text}: {count}")


--- Abortion ---
  [GPE] united states: 32
  [GPE] china: 7
  [ORG] the world health organization: 5
  [GPE] india: 5
  [GPE] united kingdom: 3
  [GPE] switzerland: 3
  [GPE] canada: 3
  [GPE] sweden: 3
  [ORG] cdc: 3
  [ORG] the guttmacher institute: 3
  [NORP] hindu: 3
  [ORG] the catholic church: 3
  [NORP] catholic: 3
  [ORG] planned parenthood: 2
  [LOC] europe: 2

--- Al-Qaeda ---
  [ORG] al-qaeda: 256
  [GPE] united states: 76
  [PERSON] bin laden: 68
  [NORP] muslim: 40
  [NORP] american: 31
  [NORP] islamic: 26
  [LOC] kashmir: 24
  [GPE] afghanistan: 21
  [GPE] pakistan: 20
  [GPE] iraq: 19
  [ORG] al-zawahiri: 15
  [ORG] taliban: 14
  [NORP] islamist: 11
  [NORP] indian: 11
  [GPE] yemen: 11


In [ ]:
def get_sentences_with_entity_multi(sentences, surface_forms):
    if isinstance(surface_forms, str):
        surface_forms = {surface_forms}
    alt = "|".join(re.escape(f.lower()) for f in surface_forms)
    pattern = re.compile(r'(?<!\w)(?:' + alt + r')(?!\w)')
    return [s for s in sentences if pattern.search(s.lower())]

# Test
for i, entity in [(0, {"united states"}), (2, {"al-qaeda"})]:
    sents = get_sentences_with_entity_multi(df['sentences'][i], entity)
    print(f"\n--- {df['title'][i]} | '{entity}' ---")
    print(f"Frases que la contienen: {len(sents)}")
    print("Ejemplo:", sents[0] if sents else "ninguna")


--- Abortion | '{'united states'}' ---
Frases que la contienen: 19
Ejemplo: The regimen (200 mg of mifepristone, followed 24–48 hours later by 800 mcg of vaginal misoprostol) previously used by Planned Parenthood clinics in the United States from 2001 to March 2006 was 98.5% effective through 63 days gestation—with an ongoing pregnancy rate of about 0.5%, and an additional 1% of women having uterine evacuation for various reasons, including problematic bleeding, persistent gestational sac, clinician judgment or a woman's request.

--- Al-Qaeda | '{'al-qaeda'}' ---
Frases que la contienen: 225
Ejemplo: Al-Qaeda is a pan-Islamist militant organization led by Sunni Islamist jihadists who self-identify as a vanguard spearheading a global Islamist revolution to unite the Muslim world under a supra-national Islamic caliphate.


# Aspect-oriented sentiment analysis

In [ ]:
absa_pipeline = pipeline(
    "text-classification",
    model="yangheng/deberta-v3-base-absa-v1.1",
    top_k=None
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [ ]:
def aggregate_sentiment_absa(results):
    if not results:
        return None
    labels = [r['label'] for r in results]
    total = len(labels)
    neg_scores = [r['scores'].get('Negative', 0) for r in results]
    pos_scores = [r['scores'].get('Positive', 0) for r in results]
    neu_scores = [r['scores'].get('Neutral', 0) for r in results]
    return {
        'n_sentences': total,
        'NEG_pct': labels.count('Negative') / total,
        'NEU_pct': labels.count('Neutral') / total,
        'POS_pct': labels.count('Positive') / total,
        'mean_NEG': np.mean(neg_scores),
        'mean_POS': np.mean(pos_scores),
        'mean_NEU': np.mean(neu_scores),
        'polarity': np.mean(pos_scores) - np.mean(neg_scores)
    }

def analyze_top_entities_absa(row, top_n=5, max_sentences=50, seed=42):
    entity_results = {}
    top_entities = row['entity_counts_norm'][:top_n]
    surface_map = get_surface_forms(row['entities'])

    for item in top_entities:
        (text, label) = item[0]
        count = item[1]

        surface_forms = surface_map.get((text, label), {text})
        sents = get_sentences_with_entity_multi(row['sentences'], surface_forms)
        if not sents:
            continue

        if len(sents) > max_sentences:
            rng = random.Random(seed)
            sents = rng.sample(sents, max_sentences)

        results = []
        for sent in sents:
            try:
                pred = absa_pipeline(sent[:480], text_pair=text)[0]
                scores = {r['label']: r['score'] for r in pred}
                label_pred = max(scores, key=scores.get)
                results.append({'label': label_pred, 'scores': scores})
            except Exception as e:
                continue

        entity_results[(text, label)] = {
            'count': count,
            'sentiment': aggregate_sentiment_absa(results)
        }

    return entity_results

# Manual test
test_result = analyze_top_entities_absa(df.iloc[0])
print("=== Abortion ===")
for (text, label), data in test_result.items():
    s = data['sentiment']
    if s:
        print(f"  [{label}] {text} (n={s['n_sentences']}): polarity={s['polarity']:.3f} | NEG={s['NEG_pct']:.0%} NEU={s['NEU_pct']:.0%} POS={s['POS_pct']:.0%}")

=== Abortion ===
  [GPE] united states (n=31): polarity=-0.134 | NEG=13% NEU=87% POS=0%
  [GPE] china (n=7): polarity=-0.067 | NEG=0% NEU=100% POS=0%
  [ORG] the world health organization (n=5): polarity=-0.034 | NEG=0% NEU=100% POS=0%
  [GPE] india (n=5): polarity=-0.060 | NEG=0% NEU=100% POS=0%
  [GPE] united kingdom (n=3): polarity=-0.007 | NEG=0% NEU=100% POS=0%


In [ ]:
tqdm.pandas()
df['absa'] = df.progress_apply(analyze_top_entities_absa, axis=1)

absa_serializable = []
for row_absa in df['absa']:
    row_dict = {}
    for (text, label), data in row_absa.items():
        row_dict[f"{text}||{label}"] = data
    absa_serializable.append(row_dict)

# Verification
print("Total processed:", df['absa'].apply(lambda x: len(x) > 0).sum())

print("\n=== Al-Qaeda ===")
for (text, label), data in df['absa'][2].items():
    s = data['sentiment']
    if s:
        print(f"  [{label}] {text} (n={s['n_sentences']}): polarity={s['polarity']:.3f} | NEG={s['NEG_pct']:.0%} NEU={s['NEU_pct']:.0%} POS={s['POS_pct']:.0%}")

100%|██████████| 80/80 [05:10<00:00,  3.88s/it]

Total processed: 80

=== Al-Qaeda ===
  [ORG] al-qaeda (n=50): polarity=-0.309 | NEG=40% NEU=56% POS=4%
  [GPE] united states (n=50): polarity=-0.381 | NEG=44% NEU=52% POS=4%
  [PERSON] bin laden (n=50): polarity=-0.238 | NEG=34% NEU=62% POS=4%
  [NORP] muslim (n=36): polarity=-0.120 | NEG=17% NEU=78% POS=6%
  [NORP] american (n=32): polarity=-0.555 | NEG=59% NEU=41% POS=0%


In [ ]:
rows = []
for idx, row in df.iterrows():
    for (entity_text, entity_label), data in row['absa'].items():
        s = data['sentiment']
        if s:
            rows.append({
                'article_id': row['id'],
                'title': row['title'],
                'entity': entity_text,
                'entity_label': entity_label,
                'entity_count': data['count'],
                'n_sentences': s['n_sentences'],
                'polarity': s['polarity'],
                'NEG_pct': s['NEG_pct'],
                'NEU_pct': s['NEU_pct'],
                'POS_pct': s['POS_pct'],
                'mean_NEG': s['mean_NEG'],
                'mean_POS': s['mean_POS'],
            })

absa_df = pd.DataFrame(rows)
print(absa_df.shape)
print(absa_df.head(10).to_string())

(400, 12)
   article_id               title                         entity entity_label  entity_count  n_sentences  polarity   NEG_pct   NEU_pct   POS_pct  mean_NEG  mean_POS
0         765            Abortion                  united states          GPE            32           31 -0.134444  0.129032  0.870968  0.000000  0.143043  0.008598
1         765            Abortion                          china          GPE             7            7 -0.067326  0.000000  1.000000  0.000000  0.084804  0.017478
2         765            Abortion  the world health organization          ORG             5            5 -0.034389  0.000000  1.000000  0.000000  0.114559  0.080170
3         765            Abortion                          india          GPE             5            5 -0.059840  0.000000  1.000000  0.000000  0.091690  0.031850
4         765            Abortion                 united kingdom          GPE             3            3 -0.007427  0.000000  1.000000  0.000000  0.011617  0.004190


## Saving data

In [ ]:
df.to_pickle("04_df_with_absa.pkl")
with open("04_absa_results.json", "w") as f:
    json.dump(absa_serializable, f, ensure_ascii=False, indent=2)

absa_df.to_csv("04_absa_summary.csv", index=False)
absa_df.to_pickle("04_absa_summary.pkl")

Mounted at /content/drive


'/content/drive/MyDrive/03_absa_summary.pkl'